In [2]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

C:\Users\siddi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_json("SEM 6\DS Lab\data_503986.json")  # or .csv depending on download

print(df.head())
print(df.shape)

                    id                                               text
0  1596647314030231552  @DonutOperator @elonmusk @stillgray It's fiery...
1  1596647313887346689  @SenMarkey @elonmusk Anti-freedom is anti-Amer...
2  1596647313719853056  @FoxNews Elon Musk voices support for Trump ri...
3  1596647313346215941  @elonmusk @CollinRugg Having meetings about me...
4  1596647312746754048  @GregA06555436 @elonmusk @TimRunsHisMouth Yes!...
(503986, 2)


In [4]:
df = df.sample(20000, random_state=42).reset_index(drop=True)

print("Dataset size:", df.shape)

Dataset size: (20000, 2)


In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

df.head()

,id,text,clean_text
0,1603474811308613632,"@BR00KLYFN_ Hello, @TwitterSupport @Twitter @e...",hello user is evading permanent suspension the...
1,1603936647644602369,@elonmusk can we fedpost on here now ?,can we fedpost on here now
2,1598328975327399936,@elonmusk One mtg with Apple and everything is...,one mtg with apple and everything is okidoki
3,1604592249001881600,@Victorshi2020 @elonmusk Who’s we??,whos we
4,1605727165412265985,@nlyonne LMAO i love you. just follow her @elo...,lmao i love you just follow her


In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="english",
    calculate_probabilities=True,
    verbose=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3359.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="english",
    calculate_probabilities=True,
    verbose=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10106.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
docs = df["clean_text"].tolist()

topics, probs = topic_model.fit_transform(docs)

2026-04-10 10:46:48,189 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 625/625 [01:36<00:00,  6.47it/s]
2026-04-10 10:48:24,932 - BERTopic - Embedding - Completed ✓
2026-04-10 10:48:24,933 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-10 10:49:04,906 - BERTopic - Dimensionality - Completed ✓
2026-04-10 10:49:04,908 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-10 10:50:43,763 - BERTopic - Cluster - Completed ✓
2026-04-10 10:50:43,778 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-10 10:50:44,290 - BERTopic - Representation - Completed ✓


In [9]:
df["topic"] = topics

df.head()

,id,text,clean_text,topic
0,1603474811308613632,"@BR00KLYFN_ Hello, @TwitterSupport @Twitter @e...",hello user is evading permanent suspension the...,46
1,1603936647644602369,@elonmusk can we fedpost on here now ?,can we fedpost on here now,272
2,1598328975327399936,@elonmusk One mtg with Apple and everything is...,one mtg with apple and everything is okidoki,30
3,1604592249001881600,@Victorshi2020 @elonmusk Who’s we??,whos we,-1
4,1605727165412265985,@nlyonne LMAO i love you. just follow her @elo...,lmao i love you just follow her,29


In [17]:
topic_model.visualize_topics()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([[0, 'twitter | tweet | tweets | social | retweet', 487],
                                   [1, 'musk | elon | via | his | jobs', 334],
                                   [2, 'tesla | electric | cars | car | ev', 299],
                                   ...,
                                   [317, 'debt | college | forgiveness | military | student', 11],
                                   [318, 'files | messhow | slates | twitterfiles | read', 10],
                                   [319, 'mom | ur | mother | loved | bdeyour', 10]],
                                  shape=(320, 3), dtype=object),
              'hovertemplate': '<b>Topic %{customdata[0]}</b><br>%{customdata[1]}<br>Size: %{customdata[2]}',
              'legendgroup': '',
              'marker': {'color': '#B0BEC5',
                         'line': {'color': 'DarkSlateGrey', 'width': 2},
                         'size': {'bdata': ('5wFOASsBFQHnAMgAswCaAIUAhQCBAH' ... 'ALAAsACwALAAsACwALAAsACgAKAA=='),
                                  'dtype': 'i2'},
                         'sizemode': 'area',
                         'sizeref': 0.304375,
                         'symbol': 'circle'},
              'mode': 'markers',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('r+drQJc7EEEoPidB0CNOQMibZ8DK6l' ... 'LBUT2AQVjf8UDWZDFBKld2QHCj3sA='),
                    'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': ('I5uJQdOyFkGLwSHBPawpwOe7BkGO0g' ... 'q+GsZmQdBLmj89tWdBBZSLQb3TAUE='),
                    'dtype': 'f4'},
              'yaxis': 'y'}],
    'layout': {'annotations': [{'showarrow': False,
                                'text': 'D1',
                                'x': np.float32(-12.04),
                                'y': np.float32(3.9771771),
                                'yshift': 10},
                               {'showarrow': False,
                                'text': 'D2',
                                'x': np.float32(7.6535316),
                                'xshift': 10,
                                'y': np.float32(23.491451)}],
               'height': 650,
               'hoverlabel': {'bgcolor': 'white', 'font': {'family': 'Rockwell', 'size': 16}},
               'legend': {'itemsizing': 'constant', 'tracegroupgap': 0},
               'margin': {'t': 60},
               'shapes': [{'line': {'color': '#CFD8DC', 'width': 2},
                           'type': 'line',
                           'x0': np.float32(7.6535316),
                           'x1': np.float32(7.6535316),
                           'y0': np.float32(-15.537097),
                           'y1': np.float32(23.491451)},
                          {'line': {'color': '#9E9E9E', 'width': 2},
                           'type': 'line',
                           'x0': np.float32(-12.04),
                           'x1': np.float32(27.347063),
                           'y0': np.float32(3.9771771),
                           'y1': np.float32(3.9771771)}],
               'sliders': [{'active': 0,
                            'pad': {'t': 50},
                            'steps': [{'args': [{'marker.color': [['red',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
   

In [11]:
topic_model.visualize_documents(docs)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'text',
              'hovertext': array(['lmao democracy is like when only the elite can vote for the rest of the population this is the trash elon is associated with now nice job',
                                  'i was against this when it first happened i didnt know the whole story i thought and still do think that we need our clandestine ops however not to turn that spy machine inward thats a difficult task and unmanageable if there are open borders',
                                  'how long before the corrupt biden admin starts screwing with space x',
                                  ...,
                                  'yall subscribe to our youtube channel this makes it easy for you to get notified once theres an update this is also part of support and god bless you as you do',
                                  'i like it hive mind', None], shape=(8044,), dtype=object),
              'marker': {'color': '#CFD8DC', 'opacity': 0.5, 'size': 5},
              'mode': 'markers+text',
              'name': 'other',
              'showlegend': False,
              'type': 'scattergl',
              'x': {'bdata': ('r69uQT8wEUHoZhZBl8wPQWG/7EAd7P' ... 'EJ4CVB9HsSQTZ7TEEyt+lAqPImQQ=='),
                    'dtype': 'f4'},
              'y': {'bdata': ('/MOpQFGNAEFi0wpBT8nyQGZ5JEA8UB' ... 'DJKvhAS9wvQKpS+EBsipZAOxW+QA=='),
                    'dtype': 'f4'}},
             {'hoverinfo': 'text',
              'hovertext': array(['so tell me twitter isnt a cesspool', 'twitter grows wilder by the day',
                                  'is timecode t necessary for twitter links that are shared not a big deal but my brain tweeks whenever i see s t',
                                  ..., 'twitters become more alive with',
                                  'twitter lost most of its advertisers fired tons of people has no communication department and is not even paying the bills for the offices also its now making red numbers so who is blind again',
                                  None], shape=(488,), dtype=object),
              'marker': {'opacity': 0.5, 'size': 5},
              'mode': 'markers+text',
              'name': '0_twitter_tweet_tweets',
              'text': array(['', '', '', ..., '', '', '0_twitter_tweet_tweets'],
                            shape=(488,), dtype=object),
              'textfont': {'size': 12},
              'type': 'scattergl',
              'x': {'bdata': ('Efo/QQctQ0HuQ0RBdEFpQYVpQUExTU' ... 'hBT+49Qe3hQ0GkE0JBSOs8QYvjQUE='),
                    'dtype': 'f4'},
              'y': {'bdata': ('nW8EQZckBUFpyQNBcmG3QCy8BEGQ9g' ... 'RBNpcFQarBBEHhEwZB5XYJQTx2A0E='),
                    'dtype': 'f4'}},
             {'hoverinfo': 'text',
              'hovertext': array(['elon musk is fat', 'james woods elon musk may have saved america via',
                                  'damn dude theres no reason on earth that he would lie none whatsoever the only thing sadder than a donald trump dick rider is an elon musk stan',
                                  ...,
                                  'elon musk has balls elon we all know theyve got dirt on you otherwise theres no way you could have become as big as you are today no of us want to be murdered amp our planet destroyed to satisfy their perverse desires fearlessly expose the bastards',
                                  'common elon musk w', None], shape=(335,), dtype=object),
              'marker': {'opacity': 0.5, 'size': 5},
              'mode': 'markers+text',
              'name': '1_musk_elon_via',
              'text': array(['', '', '', ..., '', '', '1_musk_elon_via'], shape=(335,), dtype=object),
              'textfont': {'size': 12},
              'type': 'scattergl',
              'x': {'bdata': ('QFE/QdczO0GkhjlBvHc/QdWDPEGO9j' ... '9BamU9QaIfQEGMaT9BzmpAQefAPEE='),
                    'dtype': 'f4'},
              'y': {'bdata': ('KkosQaqSKkHNhidBNIgsQVEYK0FhPi' ... '5BNFAnQYhnLUE2ZyxBN+svQX8KK

In [12]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

   Topic  Count                              Name  \
0     -1   8043                  -1_he_to_the_and   
1      0    487     0_twitter_tweet_tweets_social   
2      1    334               1_musk_elon_via_his   
3      2    299         2_tesla_electric_cars_car   
4      3    277         3_elon_elons_elonmusk_him   
5      4    231      4_jail_crime_law_accountable   
6      5    200  5_covid_vaccine_vaccines_doctors   
7      6    179               6_she_her_shes_lady   
8      7    154  7_speech_free_freedom_absolutist   
9      8    133               8_truth_lie_la_lies   

                                      Representation  \
0  [he, to, the, and, you, that, is, of, it, twit...   
1  [twitter, tweet, tweets, social, retweet, leav...   
2  [musk, elon, via, his, jobs, hero, is, he, toi...   
3  [tesla, electric, cars, car, ev, teslas, semi,...   
4  [elon, elons, elonmusk, him, he, her, riding, ...   
5  [jail, crime, law, accountable, prison, police...   
6  [covid, vaccine, vacc

In [13]:
topic_model.get_topic(0)

[('twitter', np.float64(0.017670890380011368)),
 ('tweet', np.float64(0.015425364306968772)),
 ('tweets', np.float64(0.013876029947556054)),
 ('social', np.float64(0.006305381519726287)),
 ('retweet', np.float64(0.006134630818994385)),
 ('leave', np.float64(0.005072308758320387)),
 ('my', np.float64(0.004328398477062994)),
 ('im', np.float64(0.004289796680498572)),
 ('tweeting', np.float64(0.004237302236733329)),
 ('see', np.float64(0.00408825652060569))]

In [14]:
topic_labels = topic_model.generate_topic_labels()

topic_labels[:10]

['-1_he_to_the',
 '0_twitter_tweet_tweets',
 '1_musk_elon_via',
 '2_tesla_electric_cars',
 '3_elon_elons_elonmusk',
 '4_jail_crime_law',
 '5_covid_vaccine_vaccines',
 '6_she_her_shes',
 '7_speech_free_freedom',
 '8_truth_lie_la']

In [15]:
topic_model.visualize_barchart(top_n_topics=10)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'marker': {'color': '#D55E00'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.006134630818994385, 0.006305381519726287,
                    0.013876029947556054, 0.015425364306968772,
                    0.017670890380011368],
              'xaxis': 'x',
              'y': [retweet  , social  , tweets  , tweet  , twitter  ],
              'yaxis': 'y'},
             {'marker': {'color': '#0072B2'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.004872656485259453, 0.0050459800002739365,
                    0.005613104338976989, 0.023419355229159708,
                    0.033047026292743746],
              'xaxis': 'x2',
              'y': [jobs  , his  , via  , elon  , musk  ],
              'yaxis': 'y2'},
             {'marker': {'color': '#CC79A7'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.009976552542794373, 0.012557698114034167,
                    0.013116462501053878, 0.013384876611814872, 0.0358549861293239],
              'xaxis': 'x3',
              'y': [ev  , car  , cars  , electric  , tesla  ],
              'yaxis': 'y3'},
             {'marker': {'color': '#E69F00'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.005480526922357612, 0.00617681028074958,
                    0.006419331592159521, 0.018683632092478815,
                    0.027491179451231067],
              'xaxis': 'x4',
              'y': [he  , him  , elonmusk  , elons  , elon  ],
              'yaxis': 'y4'},
             {'marker': {'color': '#56B4E9'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.011721525217426027, 0.012340883609933564,
                    0.013623445515403659, 0.014165632472438319,
                    0.016042742067159534],
              'xaxis': 'x5',
              'y': [prison  , accountable  , law  , crime  , jail  ],
              'yaxis': 'y5'},
             {'marker': {'color': '#009E73'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.009349160572295262, 0.009973950854455476,
                    0.014360242210823956, 0.021305960290361244,
                    0.03283237970442124],
              'xaxis': 'x6',
              'y': [flu  , doctors  , vaccines  , vaccine  , covid  ],
              'yaxis': 'y6'},
             {'marker': {'color': '#F0E442'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.006840829919892649, 0.00803989202048173,
                    0.023162103378510292, 0.05313287100229882, 0.06608691330962368],
              'xaxis': 'x7',
              'y': [elizabeth  , lady  , shes  , her  , she  ],
              'yaxis': 'y7'},
             {'marker': {'color': '#D55E00'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.00637757704544057, 0.008100579731203912,
                    0.027147420323958376, 0.046028858988196086,
                    0.05858330599176824],
              'xaxis': 'x8',
              'y': [expression  , absolutist  , freedom  , free  , speech  ],
              'yaxis': 'y8'},
             {'marker': {'color': '#0072B2'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.015365149342776921, 0.017636964097657554,
                    0.01875332986804404, 0.020925178419960895, 0.05854523845383085],
              'xaxis': 'x9',
              'y': [liar  , lies  , la  , lie  , truth  ],
              'yaxis': 'y9'},
             {'marker': {'color': '#CC79A7'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.007571617703666305, 0.012392373016094591,
                    0.014281630549823822, 0.015919758684228635,
                    0.021467357469068048],
              'xaxis': 'x10',
              'y': [tweeter  , musks  , elon  , twitter  , musk  ],
              'yaxis': 'y10'}],
   